# Gold: estrella de facturacion

Ejecuta `sql/gold/billing.sql`: `dim_customer` (con bridge `student_id` hacia `dim_student`), `dim_product` + `fact_invoice`, `fact_invoice_item`, `fact_payment`, `fact_subscription`.

**Requiere que `02_estrella_academica.ipynb` ya haya corrido** (el bridge de `dim_customer` depende de `gold.dim_student`).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from pathlib import Path
from utils.db import get_psycopg2_connection, get_engine

engine = get_engine()
SQL_GOLD = Path("/home/jovyan/work/sql/gold")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

In [2]:
run_sql_file(SQL_GOLD / "billing.sql")

OK: billing.sql ejecutado


## 1. Conteos: gold vs. silver (deben coincidir exacto)

In [3]:
pd.read_sql("""
    SELECT 'dim_customer' t, (SELECT count(*) FROM silver.billing__customers) silver, (SELECT count(*) FROM gold.dim_customer) gold
    UNION ALL SELECT 'dim_product', (SELECT count(*) FROM silver.billing__products), (SELECT count(*) FROM gold.dim_product)
    UNION ALL SELECT 'fact_invoice', (SELECT count(*) FROM silver.billing__invoices), (SELECT count(*) FROM gold.fact_invoice)
    UNION ALL SELECT 'fact_invoice_item', (SELECT count(*) FROM silver.billing__invoice_items), (SELECT count(*) FROM gold.fact_invoice_item)
    UNION ALL SELECT 'fact_payment', (SELECT count(*) FROM silver.billing__payments), (SELECT count(*) FROM gold.fact_payment)
    UNION ALL SELECT 'fact_subscription', (SELECT count(*) FROM silver.billing__subscriptions), (SELECT count(*) FROM gold.fact_subscription)
""", engine)

,t,silver,gold
0,dim_customer,10000,10000
1,dim_product,200,200
2,fact_invoice,50000,50000
3,fact_invoice_item,150000,150000
4,fact_payment,80000,80000
5,fact_subscription,15000,15000


## 2. Verificar el bridge estudiante <-> cliente

In [4]:
pd.read_sql("""
    SELECT is_student, count(*) FROM gold.dim_customer GROUP BY is_student
""", engine)

,is_student,count
0,False,5000
1,True,5000


## 3. Pregunta de negocio: ingreso por producto (via `invoice_items`)

In [5]:
pd.read_sql("""
    SELECT p.category, p.name,
           sum(ii.line_total) AS ingreso_total,
           count(*) AS lineas
    FROM gold.fact_invoice_item ii
    JOIN gold.dim_product p ON p.product_id = ii.product_id
    GROUP BY p.category, p.name
    ORDER BY ingreso_total DESC
    LIMIT 10
""", engine)

,category,name,ingreso_total,lineas
0,premium,Product 00122,200950.45,824
1,basic,Product 00069,199589.60,791
2,standard,Product 00144,198777.52,788
3,standard,Product 00151,194153.70,798
4,basic,Product 00061,192425.45,774
5,standard,Product 00135,191767.80,803
6,basic,Product 00129,191311.75,769
7,basic,Product 00104,191295.92,772
8,premium,Product 00112,188903.70,848
9,standard,Product 00157,188470.22,791


## 4. Pregunta de negocio: churn (cancelaciones) por segmento de cliente

In [6]:
pd.read_sql("""
    SELECT c.segment,
           count(*) AS suscripciones,
           count(*) FILTER (WHERE s.is_cancelled) AS canceladas,
           round(100.0 * count(*) FILTER (WHERE s.is_cancelled) / count(*), 1) AS pct_churn
    FROM gold.fact_subscription s
    JOIN gold.dim_customer c ON c.customer_id = s.customer_id
    GROUP BY c.segment
    ORDER BY pct_churn DESC
""", engine)

,segment,suscripciones,canceladas,pct_churn
0,retail,10528,1595,15.2
1,smb,3301,494,15.0
2,enterprise,1171,153,13.1


## 5. Cruce cross-domain: retencion de clientes que ademas son estudiantes vs. los que no

In [7]:
pd.read_sql("""
    SELECT c.is_student,
           count(*) AS suscripciones,
           round(100.0 * count(*) FILTER (WHERE s.is_active) / count(*), 1) AS pct_activas,
           round(100.0 * count(*) FILTER (WHERE s.is_cancelled) / count(*), 1) AS pct_canceladas
    FROM gold.fact_subscription s
    JOIN gold.dim_customer c ON c.customer_id = s.customer_id
    GROUP BY c.is_student
""", engine)

,is_student,suscripciones,pct_activas,pct_canceladas
0,False,7427,74.9,15.4
1,True,7573,75.4,14.5
